# Notebook Arsitektur Model + Worker Runtime Resource Check

Notebook ini hanya mencakup **arsitektur model** dan **setelan resource worker**.

Cakupan:
1. Konfigurasi arsitektur classical dan hybrid.
2. Definisi model dari file `model_architecture_modules.py`.
3. Deteksi CPU/GPU, RAM/VRAM, dan jumlah core dari `worker_runtime_config.py`.
4. Rekomendasi `micro_batch_size`, `gradient_accumulation_steps`, `num_workers`, `pin_memory`, dan precision.
5. Tabel layer, parameter, parameter group, serta gambar arsitektur.

Yang sengaja tidak dimasukkan di notebook ini:
- training loop final,
- evaluasi metrik akhir,
- XAI/Grad-CAM/Integrated Gradients.

XAI ditempatkan di notebook evaluasi akhir setelah checkpoint terbaik tersedia.

## 1. Import dan folder artefak

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import torch

def find_project_root(marker="dataset_final_manifest.csv"):
    """Cari root repo dari current working directory notebook."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Root proyek dengan {marker!r} tidak ditemukan dari {start} atau parent-nya.")

PROJECT_ROOT = find_project_root()
MODULE_PATH = PROJECT_ROOT / "03_model_architecture"
if str(MODULE_PATH.resolve()) not in sys.path:
    sys.path.append(str(MODULE_PATH.resolve()))

from model_architecture_settings import default_architecture_config
from model_architecture_modules import (
    ClassicalFullySpatialCNN,
    HybridQCQCNN,
    count_parameters,
    parameter_table,
    make_classical_parameter_groups,
    make_hybrid_parameter_groups,
    parameter_group_summary,
    forward_layer_summary,
    quantum_circuit_detail_table,
    regularization_trainability_table,
    make_architecture_block_diagram,
    make_quantum_ring_circuit_fallback,
    make_module_tree_text,
    make_text_image,
)
from worker_runtime_config import (
    WorkerRuntimeProfile,
    detect_worker_resources,
    build_runtime_plan,
    export_runtime_audit,
    print_runtime_summary,
)

ARTIFACT_DIR = PROJECT_ROOT / "architecture_worker_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR

## 2. Konfigurasi arsitektur yang dikunci

Konfigurasi ini memuat input 128×128 grayscale, shared CNN backbone, classical head, dan hybrid quantum head. Untuk hybrid: 8 qubit, amplitude encoding, Strongly Entangling Layers, depth 2, dan ring/circular entanglement pada diagram dokumentasi.

In [ ]:
cfg = default_architecture_config()
print(json.dumps(cfg.to_dict(), indent=2))

with open(ARTIFACT_DIR / "architecture_config.json", "w", encoding="utf-8") as f:
    json.dump(cfg.to_dict(), f, indent=2, ensure_ascii=False)

## 3. Worker runtime profile

Setelan ini dapat diubah per worker node.

Contoh skenario:
- `torch_mode='auto'`: otomatis memakai GPU jika tersedia, kalau tidak CPU.
- `torch_mode='gpu'` dan `strict=True`: wajib GPU; error kalau GPU tidak tersedia.
- `quantum_mode='auto'`: otomatis memilih `lightning.gpu` jika GPU/PennyLane tersedia, lalu fallback ke CPU.
- `target_effective_batch_size`: batch efektif yang ingin dicapai melalui gradient accumulation.

`recommended_micro_batch_size × gradient_accumulation_steps = effective_batch_size`

In [ ]:
worker_profile = WorkerRuntimeProfile(
    worker_id="architecture_resource_check",
    torch_mode="auto",          # auto | gpu | cpu
    quantum_mode="auto",        # auto | gpu | cpu
    strict=False,
    target_effective_batch_size=32,
    min_micro_batch_size=1,
    max_micro_batch_size=64,
    safety_vram_fraction=0.70,
    safety_ram_fraction=0.55,
    q_diff_method="adjoint",
    dataloader_worker_cap=8,
    reserve_ram_gib=2.0,
    dry_run_batch_probe=False,
)
worker_profile

## 4. Deteksi CPU/GPU, RAM/VRAM, core, dan dependency quantum

In [ ]:
resources = detect_worker_resources(worker_profile)
resources_df = pd.DataFrame([resources.to_dict()])
resources_df.to_csv(ARTIFACT_DIR / "detected_worker_resources.csv", index=False)
resources_df

## 5. Rencana runtime classical dan hybrid

Bagian ini menghitung rekomendasi resource agar training tidak mudah crash:

- `torch_device`: CPU/GPU yang dipakai.
- `recommended_micro_batch_size`: batch kecil yang aman untuk satu forward/backward.
- `gradient_accumulation_steps`: jumlah akumulasi gradien agar batch efektif tercapai.
- `effective_batch_size`: batch efektif aktual.
- `dataloader_num_workers`: jumlah worker DataLoader yang disarankan berdasarkan core dan RAM.
- `pin_memory` dan `persistent_workers`: mengikuti device dan jumlah worker.
- `precision`: AMP diaktifkan jika CUDA tersedia.

In [ ]:
classical_plan = build_runtime_plan("classical", worker_profile, resources)
hybrid_plan = build_runtime_plan("hybrid", worker_profile, resources)
plans = [classical_plan, hybrid_plan]

print_runtime_summary(resources, plans)
export_runtime_audit(resources, plans, ARTIFACT_DIR)
plans_df = pd.DataFrame([p.to_dict() for p in plans])
plans_df

## 6. Inisialisasi model classical dan hybrid

Catatan: jika PennyLane/Lightning GPU belum tersedia, hybrid akan memakai surrogate head hanya untuk **smoke test arsitektur dan tabel parameter** agar notebook tidak crash. Pada training final, environment quantum harus memasang PennyLane + backend Lightning yang sesuai.

In [ ]:
classical_model = ClassicalFullySpatialCNN(cfg)

hybrid_model = HybridQCQCNN(
    cfg,
    q_device=hybrid_plan.quantum_device,
    q_diff_method=hybrid_plan.quantum_diff_method,
    q_device_fallbacks=worker_profile.q_device_preference_gpu if hybrid_plan.torch_device.startswith("cuda") else worker_profile.q_device_preference_cpu,
)

print("Classical model:", classical_model.__class__.__name__)
print("Hybrid model:", hybrid_model.__class__.__name__)
print("Hybrid quantum runtime:", getattr(hybrid_model, "quantum_runtime", "unknown"))

## 7. Smoke test output shape

In [ ]:
input_shape = cfg.image.input_shape_chw
x = torch.randn(2, *input_shape)
with torch.no_grad():
    y_classical = classical_model(x)
    y_hybrid = hybrid_model(x)

print("Input shape      :", tuple(x.shape))
print("Classical output :", tuple(y_classical.shape))
print("Hybrid output    :", tuple(y_hybrid.shape))

## 8. Tabel parameter model

In [ ]:
parameter_comparison = pd.DataFrame([
    {"model": "classical_fully_spatial_cnn", **count_parameters(classical_model)},
    {"model": "hybrid_qcqcnn", **count_parameters(hybrid_model), "quantum_runtime": getattr(hybrid_model, "quantum_runtime", "unknown")},
])
parameter_comparison.to_csv(ARTIFACT_DIR / "architecture_parameter_comparison.csv", index=False)
parameter_comparison

In [ ]:
classical_parameter_table = pd.DataFrame(parameter_table(classical_model))
hybrid_parameter_table = pd.DataFrame(parameter_table(hybrid_model))

classical_parameter_table.to_csv(ARTIFACT_DIR / "classical_parameter_table.csv", index=False)
hybrid_parameter_table.to_csv(ARTIFACT_DIR / "hybrid_parameter_table.csv", index=False)

print("Classical parameter table saved:", ARTIFACT_DIR / "classical_parameter_table.csv")
print("Hybrid parameter table saved   :", ARTIFACT_DIR / "hybrid_parameter_table.csv")
hybrid_parameter_table.head(20)

## 9. Parameter group learning rate

In [ ]:
classical_groups = make_classical_parameter_groups(classical_model, cfg)
hybrid_groups = make_hybrid_parameter_groups(hybrid_model, cfg)

classical_group_summary = pd.DataFrame(parameter_group_summary(classical_groups))
hybrid_group_summary = pd.DataFrame(parameter_group_summary(hybrid_groups))

classical_group_summary.to_csv(ARTIFACT_DIR / "classical_parameter_group_summary.csv", index=False)
hybrid_group_summary.to_csv(ARTIFACT_DIR / "hybrid_parameter_group_summary.csv", index=False)

print("Classical groups")
display(classical_group_summary)
print("Hybrid groups")
display(hybrid_group_summary)

## 10. Tabel layer forward summary

In [ ]:
classical_forward_summary = pd.DataFrame(forward_layer_summary(classical_model, input_shape=input_shape, batch_size=2))
hybrid_forward_summary = pd.DataFrame(forward_layer_summary(hybrid_model, input_shape=input_shape, batch_size=2))

classical_forward_summary.to_csv(ARTIFACT_DIR / "classical_forward_layer_summary.csv", index=False)
hybrid_forward_summary.to_csv(ARTIFACT_DIR / "hybrid_forward_layer_summary.csv", index=False)

classical_forward_summary.head(30)

## 11. Tabel detail circuit quantum

In [ ]:
quantum_detail = pd.DataFrame(quantum_circuit_detail_table(cfg, selected_q_device=hybrid_plan.quantum_device))
quantum_detail.to_csv(ARTIFACT_DIR / "quantum_circuit_detail_table.csv", index=False)
quantum_detail

## 12. Tabel mitigasi regularisasi dan trainability

Tabel ini hanya merangkum posisi desain. Implementasi training detail dilakukan pada notebook training/worker node.

In [ ]:
mitigation_table = pd.DataFrame(regularization_trainability_table())
mitigation_table.to_csv(ARTIFACT_DIR / "regularization_trainability_mitigation_table.csv", index=False)
mitigation_table

## 13. Gambar arsitektur dan module tree PyTorch

In [ ]:
high_level_path = make_architecture_block_diagram(ARTIFACT_DIR / "high_level_classical_hybrid_architecture.png")
quantum_fallback_path = make_quantum_ring_circuit_fallback(
    ARTIFACT_DIR / "hybrid_qcqcnn_quantum_circuit_fallback.png",
    n_qubits=cfg.quantum_head.n_qubits,
    depth=cfg.quantum_head.depth,
)
classical_tree_path = make_text_image(
    make_module_tree_text(classical_model),
    ARTIFACT_DIR / "classical_pytorch_module_tree.png",
    title="Classical Fully Spatial CNN - PyTorch Module Tree",
)
hybrid_tree_path = make_text_image(
    make_module_tree_text(hybrid_model),
    ARTIFACT_DIR / "hybrid_pytorch_module_tree.png",
    title="Hybrid QCQ-CNN - PyTorch Module Tree",
)

print(high_level_path)
print(quantum_fallback_path)
print(classical_tree_path)
print(hybrid_tree_path)

## 14. Helper untuk training notebook berikutnya

Contoh pemakaian rencana runtime pada training loop nanti:

```python
loader = DataLoader(
    train_dataset,
    batch_size=classical_plan.recommended_micro_batch_size,
    num_workers=classical_plan.dataloader_num_workers,
    pin_memory=classical_plan.pin_memory,
    persistent_workers=classical_plan.persistent_workers,
)

accum_steps = classical_plan.gradient_accumulation_steps

for step, batch in enumerate(loader):
    loss = model_step(batch) / accum_steps
    loss.backward()
    if (step + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
```

Dengan cara ini, worker yang VRAM/RAM-nya kecil tetap bisa mengejar batch efektif melalui gradient accumulation.

## 15. Ringkasan artefak yang dihasilkan

In [ ]:
for path in sorted(ARTIFACT_DIR.glob("*")):
    print(path)